# 🧵 Threads and the interpreter lock

*Section 14.1 of the notes.*

A **thread** is an independent sequence of instructions inside one running
program. All threads of a program share its memory: they see the same
variables and the same arrays. Your laptop has many cores, so several threads
could in principle run at the same moment, one per core. 🚀

There is a catch in Python. 🔒 The standard Python interpreter has a **global
interpreter lock** (the *GIL*): only the thread that holds the lock may execute
Python instructions. The others wait for their turn.

So when do threads help? This notebook runs four small experiments to find
out:

| Experiment | What the threads do | Do they help? |
|---|---|---|
| 1 | Python arithmetic | 🤔 let's see |
| 2 | wait (like reading a file) | 🤔 let's see |
| 3 | run compiled NumPy code | 🤔 let's see |
| 4 | finish at different times | not about speed: how to collect the results |

Before running each experiment, try to guess the answer!

## 🔒 More about the GIL

**It belongs to an interpreter, not to the language.** *Python* is a
language: a set of rules that say what a program means. A program that reads
Python code and runs it is an *implementation* of the language. The one
almost everybody uses, the one you download from python.org or get with
conda, is **CPython**, written in C. The GIL is a design choice of CPython.
Other implementations made other choices: Jython (written in Java) and
IronPython (written for .NET) never had a GIL.

**Why CPython has it.** 🧮 CPython keeps track of memory by *reference
counting*: every object stores how many names, lists, and other objects refer
to it, and the object is deleted as soon as that count drops to zero. Suppose
two threads change the count of the same object at the same moment. One of the
two changes can be lost, and then an object is deleted while it is still in
use (the program crashes) or is never deleted (memory leaks). Protecting every
object with a lock of its own is complicated and slow. One lock for the whole
interpreter is simple, keeps single-threaded programs fast, and makes it easy
to write extensions in C, such as NumPy. That ease is one reason why Python
has so many scientific libraries.

**It is being removed.** 🔓 Today's computers have many cores, and the GIL
keeps Python code on one of them at a time. In 2023 the CPython developers
accepted a plan (PEP 703) to make the GIL optional:

| Version | Released | Free-threaded build (no GIL) |
|---|---|---|
| Python 3.13 | October 2024 | first version, *experimental*, a separate install (`python3.13t`) |
| Python 3.14 | October 2025 | *officially supported*, still a separate install |
| later | | planned to become the default, once enough libraries support it |

So the GIL is going away, step by step, but the Python you install by default
still has it. Every C extension must be checked and adapted to run without the
lock, and the major scientific libraries, among them NumPy, now ship versions
for the free-threaded build. The price of the free-threaded build is that a
program with a single thread runs somewhat slower, because the reference
counts need more careful bookkeeping.

👉 In this notebook, check the line `lock enabled` printed by the setup cell.
With the usual Python it prints `True`, and the experiments below behave as
described. With a free-threaded build it prints `False`, and experiment 1
turns out differently. 🧪 If you are curious, install a free-threaded Python
(for example with `uv python install 3.14t`) and run the notebook there.

## ⚙️ Setup

The first cell prints three facts about your machine:

* **cores**: how many threads can run at the very same moment;
* **python**: the version of the interpreter;
* **lock enabled**: whether the GIL is on: `True` for the usual Python,
  `False` for a free-threaded build (see the previous cell).

⏱️ We time with `%timeit`, as in notebook `04-measuring` of Chapter 13:

* `-r 3 -n 1` runs the code 3 times, one call per run, since each call
  already takes a while;
* `-o -q` returns the result quietly instead of printing it;
* `.best` is the **smallest** of the three times, the one with the least
  disturbance. Remember that the first run of a computation is often slower.

The functions that the threads run live in the file `tasks.py`, next to this
notebook.

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np

import tasks

print("cores:", os.cpu_count())
print("python:", sys.version.split()[0])
print("lock enabled:", getattr(sys, "_is_gil_enabled", lambda: True)())

## 1. 🐍 Python arithmetic gains nothing

`tasks.cpu_work(n)` is a plain Python loop that adds up `(i * i) % 97` for
`n` values of `i`. We take `n` = 20 million, about half a second of work. Every step is a Python instruction, so the thread
running it must hold the lock the whole time.

We do this work `N = 8` times, in two ways:

1. **sequentially**: one call after the other, in a list comprehension;
2. **with 8 threads**, using a `ThreadPoolExecutor`. It works like this:
   * `ThreadPoolExecutor(max_workers=k)` starts a *pool* of `k` threads that
     wait for jobs;
   * `pool.map(f, args)` hands out one job `f(x)` for every `x` in `args`, and
     the free threads pick them up;
   * `list(...)` collects all the results;
   * leaving the `with` block waits until every job is done and shuts the
     threads down.

The **speed-up** is the sequential time divided by the threaded time. With 8
threads on a machine with at least 8 cores, the best we could hope for is 8.

🤔 *Guess before you run: what speed-up do you expect?* (The cell takes about
half a minute.)

In [ ]:
N = 8  # number of jobs
n = 20_000_000  # loop steps per job

t_seq = %timeit -o -q -r 3 -n 1 [tasks.cpu_work(n) for _ in range(N)]


def with_threads(k):
    with ThreadPoolExecutor(max_workers=k) as pool:
        list(pool.map(tasks.cpu_work, [n] * N))  # one argument per job: N copies of n


t_thr = %timeit -o -q -r 3 -n 1 with_threads(N)
t_seq, t_thr = t_seq.best, t_thr.best
print(f"sequential {t_seq:5.2f} s")
print(f"{N} threads  {t_thr:5.2f} s   speed-up {t_seq / t_thr:.2f}x")

💡 **What happened?** The speed-up is about 1: eight threads are no faster
than one. Only the thread holding the lock can run Python instructions, so the
threads take turns, and the total amount of work done per second stays the
same. The other cores sit idle. 😴

👉 For pure Python arithmetic, threads do **not** help. Notebook
`02-processes` shows what does: several processes, each with its own
interpreter and its own lock.

## 2. ⏳ Waiting gains a lot

Many programs spend their time **waiting**: for a file to be read from disk,
for a web server to answer, for a measurement device. `tasks.wait(0.2)` stands
for such a wait: it sleeps for 0.2 seconds.

A thread that waits does not execute Python instructions, so it **releases the
lock** while it waits. Think of a kitchen ☕: while one kettle heats up, you can
put the next one on the stove. You do not have to stand next to the first
kettle until it boils.

Eight waits of 0.2 s take 1.6 s one after the other.

🤔 *How long will they take with 8 threads?*

In [ ]:
t_seq = %timeit -o -q -r 3 -n 1 [tasks.wait(0.2) for _ in range(N)]


def wait_threads(k):
    with ThreadPoolExecutor(max_workers=k) as pool:
        list(pool.map(tasks.wait, [0.2] * N))


t_thr = %timeit -o -q -r 3 -n 1 wait_threads(N)
t_seq, t_thr = t_seq.best, t_thr.best
print(f"sequential {t_seq:5.2f} s")
print(f"{N} threads  {t_thr:5.2f} s   speed-up {t_seq / t_thr:.2f}x")

💡 **What happened?** All eight waits overlap, so the whole thing takes
about 0.2 s instead of 1.6 s: a speed-up of almost 8. 🎉 The small gap to
exactly 8 is the time to start the threads and hand out the jobs.

👉 Threads are the right tool when the program **waits**, for files, for the
network, or for devices.

## 3. ⚡ Work that leaves Python gains too

`tasks.elementwise(a)` computes `np.sin(a).sum()` on an array of 20 million
numbers. Here the heavy work is not done by Python instructions: `np.sin` runs
a loop written in C, inside NumPy. While that compiled loop runs, NumPy
**releases the lock**, because the loop does not touch any Python objects.

So several threads can each run their own `np.sin` at the same time, on
different cores. Each thread gets its own array, so the 8 threads work on
8 different arrays.

🤔 *Guess the speed-up before you run!*

In [ ]:
arrays = [np.random.default_rng(k).standard_normal(20_000_000) for k in range(N)]

t_seq = %timeit -o -q -r 3 -n 1 [tasks.elementwise(a) for a in arrays]


def elementwise_threads(k):
    with ThreadPoolExecutor(max_workers=k) as pool:
        list(pool.map(tasks.elementwise, arrays))


t_thr = %timeit -o -q -r 3 -n 1 elementwise_threads(N)
t_seq, t_thr = t_seq.best, t_thr.best
print(f"sequential {t_seq:5.2f} s")
print(f"{N} threads  {t_thr:5.2f} s   speed-up {t_seq / t_thr:.2f}x")

💡 **What happened?** The speed-up is large, close to the number of threads,
because the eight `np.sin` loops really run in parallel on eight cores. 🏎️

Why not exactly 8? Some parts of each job still need the lock (calling the
function, creating the result array, the `.sum()` and the `float(...)`), and
starting the threads costs a little time. Not all cores of a laptop are equally
fast either: many processors mix fast and slower cores.

⚠️ **A measurement trap.** Run the two timings with a single call each and the
threaded version can look much worse, around 2x. The very first threaded run
asks the operating system for eight fresh result arrays of 160 MB at once, and
handing over that memory is slow. This is the first-call effect of
`04-measuring`, and the reason we keep the smallest of three runs.

👉 Threads **do** help when the work happens inside compiled code that
releases the lock, as in most heavy NumPy, SciPy, and PyTorch operations.

## 4. 📬 Getting results as they finish

Jobs do not all take the same time. Here four jobs wait for 0.4, 0.1, 0.3, and
0.2 seconds. There are two ways to collect their results:

* `pool.map(f, args)` returns the results **in the order of the arguments**.
  It hands you the result of the first job first, even if a later job finished
  earlier.
* `pool.submit(f, x)` starts one job and immediately returns a **future**, a
  placeholder for a result that does not exist yet 🎟️. `as_completed(futures)`
  gives back the futures **in the order in which they finish**, and
  `f.result()` reads the result out of a finished future.

The second way is useful to show progress, or to start working on the first
results while the others are still running.

🤔 *Guess the two printed lists before you run!*

In [ ]:
delays = [0.4, 0.1, 0.3, 0.2]

with ThreadPoolExecutor(max_workers=4) as pool:
    print("map:         ", list(pool.map(tasks.wait, delays)))

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(tasks.wait, d) for d in delays]
    print("as_completed:", [f.result() for f in as_completed(futures)])

💡 **What happened?** `map` gives `[0.4, 0.1, 0.3, 0.2]`, the order of the
input, while `as_completed` gives `[0.1, 0.2, 0.3, 0.4]`: the shortest wait
finishes first. Both take about 0.4 s in total, because the four waits overlap.

⏳ **When does the `with` block end?** Leaving the `with` block waits until
every job of the pool has finished: Python does not run the next line after
the block until all the threads are done.

In the cell above this wait does not matter, because the line with
`f.result()` is inside the block, and `f.result()` itself waits for its job.
To see the wait of the `with` block, consider a variant in which the block only
starts the jobs:

```python
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(tasks.wait, d) for d in delays]
    # nothing here calls f.result()

print([f.result() for f in futures])  # all jobs have finished, no waiting
```

The timeline of this variant:

```
t = 0.0 s   submit starts the 4 jobs and returns at once
t = 0.0 s   the end of the block is reached: Python stops and waits there
t = 0.1 s   the job with 0.1 finishes
t = 0.2 s   the job with 0.2 finishes
t = 0.3 s   the job with 0.3 finishes
t = 0.4 s   the job with 0.4 finishes: Python continues after the block
t = 0.4 s   print(...) runs, and every f.result() is already there
```

So when the first line after a `with` block runs, no job is still running.

## 🎯 Summary

| Work done by the threads | Holds the lock? | Speed-up with threads |
|---|---|---|
| Python arithmetic | yes, all the time | ❌ about 1 |
| waiting (files, network) | no | ✅ about the number of threads |
| compiled code (NumPy) | no, while the C loop runs | ✅ close to the number of threads |

## 🧪 Try it yourself

1. In experiment 3, change the number of threads `k` in
   `elementwise_threads(k)` to 1, 2, 4, 8, and 16, and record the speed-up.
   Where does it stop growing? Compare with the number of cores printed in the
   setup cell.
2. In experiment 2, replace `0.2` by `0.02`. Does the speed-up stay close to
   8? Why might it drop when each wait is very short?
3. Restart the kernel and time experiment 3 with a single run, `-r 1`, instead
   of `-r 3`. Do you see the measurement trap?